In [103]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix
from statsmodels.nonparametric.smoothers_lowess import lowess

Set constants and load data

In [104]:
# set variables
threshold = 0.3
look_back = 1
look_forward_time = 24
refractory_time = 24

In [105]:
# set file path
# Make sure there are three folders: Code, Data, Results to read from and write to
file_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie"
data_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie/Data/Time_Dependent_All/Refractory"
data_save_path = os.path.join(data_path, f'Threshold_{int(threshold * 100)}_Lookforward_{look_forward_time}_Lookback_{look_back}')
os.makedirs(data_save_path, exist_ok=True)

In [106]:
# load data
with open(f"{file_path}/Data/patient.data.pkl", 'rb') as f:
    data = pickle.load(f)
patient_data = data['patient_data']
print(patient_data.shape)

(4429852, 27)


Prep Data

In [107]:
# create a list of dfs, 1 df per LCSN
patient_df_list = [group.reset_index(drop=True) for _, group in patient_data.groupby('LCSN')]
# remove dfs where patient doesn't have x hours of data
patient_df_list_filtered = [df for df in patient_df_list if len(df) >= (look_back * 4 + 1)]

Refractory Period

In [108]:
# refractory period, turn off alarms
# when alarm goes off, turn off alarm for the next x (refractory period) hours, alarms/the alarms do not go through
# alarm is if model.score >= threshold

In [109]:
refractory_period_df_list = []

for df in tqdm(patient_df_list_filtered):
    df['refractory_alarm'] = 0
    i = 0
    # if alarm at time point = 1, skip the next x hours, then continue the process
    while i < len(df):
        if df.at[i, 'model.score'] >= threshold:
            df.at[i, 'refractory_alarm'] = 1
            for j in range(i + 1, min(i + (refractory_time * 4 + 1), len(df))):
                df.at[j, 'refractory_alarm'] = 0
            i += (refractory_time * 4 + 1)
        else:
            df.at[i, 'refractory_alarm'] = 0
            i += 1
    refractory_period_df_list.append(df)

100%|██████████| 31237/31237 [01:28<00:00, 351.03it/s]


Get Look back max and look forward outcome

In [110]:
# at each time point, look at time frame x hours back, look forward y hours to see outcome
# track if alarm went off in past x hours, and outcome in next y hours
# start from time point x * 4 (since we need to look back x hours)
# loop through list of dfs
new_refractory_period_df_list = []
for patient_df in tqdm(refractory_period_df_list):
    alarm_pred_list = []
    outcome_list = []

    for time_point in patient_df['time.point']:
        if (time_point >= (look_back * 4)) and (patient_df['time.sep3.outcome'].iloc[time_point] != 1):
            alarm_pred = (patient_df.iloc[(time_point - (look_back * 4)):time_point]['refractory_alarm'] == 1).any()
            outcome = (patient_df.iloc[time_point:time_point + (look_forward_time * 4)]['time.sep3.outcome'] == 1).any()
            alarm_pred_list.append(alarm_pred)
            outcome_list.append(outcome)
        else:
            alarm_pred_list.append(-1)
            outcome_list.append(-1)
    
    new_df = patient_df.copy()
    new_df['look_back_alarm'] = alarm_pred_list
    new_df['look_forward_outcome'] = outcome_list
    new_refractory_period_df_list.append(new_df)

100%|██████████| 31237/31237 [09:20<00:00, 55.71it/s] 


In [111]:
# recombine dfs, save
combined_df = pd.concat(new_refractory_period_df_list, ignore_index=True)
# remove all rows where look_back_max or look_forward_outcome are -1
combined_df_filtered = combined_df[~((combined_df['look_back_alarm'] == -1) & (combined_df['look_forward_outcome'] == -1))]

In [112]:
# save
#combined_df_filtered.to_csv(os.path.join(data_save_path, 'Time_Dependent_Data.csv'), index = False)

GO FROM HERE FOR PLOT

In [113]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix
from statsmodels.nonparametric.smoothers_lowess import lowess

In [114]:
# # set variables
# threshold = 0.25
# look_back = 2
# look_forward_time = 8

In [115]:
# # set file path
# # Make sure there are three folders: Code, Data, Results to read from and write to
# file_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie"
# data_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie/Data/Time_Dependent_All/Refractory"
# data_save_path = os.path.join(data_path, f'Threshold_{int(threshold * 100)}_Lookforward_{look_forward_time}_Lookback_{look_back}')

In [116]:
# combined_df_filtered = pd.read_csv(os.path.join(data_save_path, 'Time_Dependent_Data.csv'))

In [117]:
# replace true in look_forward_outcome and look_back_alarm with 1 and false with 0
combined_df_filtered['look_forward_outcome'] = combined_df_filtered['look_forward_outcome'].astype(int)
combined_df_filtered['look_back_alarm'] = combined_df_filtered['look_back_alarm'].astype(int)

C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\1240577249.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['look_forward_outcome'] = combined_df_filtered['look_forward_outcome'].astype(int)
C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\1240577249.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['look_back_alarm'] = combined_df_filtered['look_back_alarm'].astype(int)


In [118]:
# create list of dfs grouped by time point
time_point_df_list = [group for _, group in combined_df_filtered.groupby('time.point')]

In [ ]:
# # get metrics at each time point no bootstrapping
# # prediction = alarm went off (max score in past x hours is >= threshold)
# # outcome = if there was sepsis in the next 8 hours or not
# import warnings
# from sklearn.exceptions import UndefinedMetricWarning

# warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# # loop through each time point
# time_point = []
# incidence = []
# tp = []
# fp = []
# tn = []
# fn = []
# auc = []
# sensitivity = []
# specificity = []
# ppv = []
# npv = []
# at_risk = []

# for df in tqdm(time_point_df_list):
#     predictions = df['look_back_alarm']
#     outcomes = df['look_forward_outcome']

#     time_point.append(df['time.point'].iloc[0])

#     incidence.append(np.mean(outcomes) * 100)
#     tp_val = np.sum((outcomes == 1) & (predictions == 1))  
#     fp_val = np.sum((outcomes == 0) & (predictions == 1))  
#     tn_val = np.sum((outcomes == 0) & (predictions == 0))  
#     fn_val = np.sum((outcomes == 1) & (predictions == 0))  
#     tp.append(tp_val)
#     fp.append(fp_val)
#     tn.append(tn_val)
#     fn.append(fn_val)

#     auc_value = roc_auc_score(outcomes, predictions)
#     auc.append(auc_value)

#     sensitivity_value = tp_val / (tp_val + fn_val) if tp_val + fn_val > 0 else 0
#     sensitivity.append(sensitivity_value)
#     specificity_value = tn_val / (tn_val + fp_val) if tn_val + fp_val > 0 else 0
#     specificity.append(specificity_value)

#     ppv_value = tp_val / (tp_val + fp_val) if tp_val + fp_val > 0 else 0
#     ppv.append(ppv_value)
#     npv_value = tn_val / (tn_val + fn_val) if tn_val + fn_val > 0 else 0
#     npv.append(npv_value)

#     at_risk.append(len(df))

In [ ]:
# bootstrapping
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from sklearn.exceptions import UndefinedMetricWarning
import warnings

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

n_boot = 500

time_point = []

auc_boot = []
sens_boot = []
spec_boot = []
ppv_boot = []
npv_boot = []
incidence_boot = []

tp_list = []
fp_list = []
tn_list = []
fn_list = []
at_risk_list = []

for df in tqdm(time_point_df_list):

    predictions = df['look_back_alarm'].values
    outcomes = df['look_forward_outcome'].values

    time_point.append(df['time.point'].iloc[0])

    n = len(df)

    tp = np.sum((outcomes == 1) & (predictions == 1))
    fp = np.sum((outcomes == 0) & (predictions == 1))
    tn = np.sum((outcomes == 0) & (predictions == 0))
    fn = np.sum((outcomes == 1) & (predictions == 0))

    tp_list.append(tp)
    fp_list.append(fp)
    tn_list.append(tn)
    fn_list.append(fn)
    at_risk_list.append(n)

    auc_samples = []
    sens_samples = []
    spec_samples = []
    ppv_samples = []
    npv_samples = []
    inc_samples = []

    for _ in range(n_boot):

        idx = np.random.randint(0, n, n)

        y_pred = predictions[idx]
        y_true = outcomes[idx]

        inc_samples.append(np.mean(y_true) * 100)

        try:
            auc_samples.append(roc_auc_score(y_true, y_pred))
        except ValueError:
            auc_samples.append(np.nan)

        tp_b = np.sum((y_true == 1) & (y_pred == 1))
        fp_b = np.sum((y_true == 0) & (y_pred == 1))
        tn_b = np.sum((y_true == 0) & (y_pred == 0))
        fn_b = np.sum((y_true == 1) & (y_pred == 0))

        sens_samples.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else np.nan)
        spec_samples.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else np.nan)
        ppv_samples.append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else np.nan)
        npv_samples.append(tn_b / (tn_b + fn_b) if (tn_b + fn_b) > 0 else np.nan)

    auc_boot.append(auc_samples)
    sens_boot.append(sens_samples)
    spec_boot.append(spec_samples)
    ppv_boot.append(ppv_samples)
    npv_boot.append(npv_samples)
    incidence_boot.append(inc_samples)

100%|██████████| 10142/10142 [25:16<00:00,  6.69it/s] 


In [ ]:
# get CIs
def ci(x, low=2.5, high=97.5):
    x = np.array(x, dtype=float)
    return np.nanpercentile(x, low), np.nanpercentile(x, high)

summary = []

for i, tp in enumerate(time_point):

    auc_ci_low, auc_ci_high = ci(auc_boot[i])
    sens_ci_low, sens_ci_high = ci(sens_boot[i])
    spec_ci_low, spec_ci_high = ci(spec_boot[i])
    ppv_ci_low, ppv_ci_high = ci(ppv_boot[i])
    npv_ci_low, npv_ci_high = ci(npv_boot[i])
    inc_ci_low, inc_ci_high = ci(incidence_boot[i])

    summary.append({
        "Time Point": tp,

        "AUC": np.nanmean(auc_boot[i]),
        "AUC CI Low": auc_ci_low,
        "AUC CI High": auc_ci_high,

        "Sensitivity": np.nanmean(sens_boot[i]),
        "Sensitivity CI Low": sens_ci_low,
        "Sensitivity CI High": sens_ci_high,

        "Specificity": np.nanmean(spec_boot[i]),
        "Specificity CI Low": spec_ci_low,
        "Specificity CI High": spec_ci_high,

        "PPV": np.nanmean(ppv_boot[i]),
        "PPV CI Low": ppv_ci_low,
        "PPV CI High": ppv_ci_high,

        "NPV": np.nanmean(npv_boot[i]),
        "NPV CI Low": npv_ci_low,
        "NPV CI High": npv_ci_high,

        "Incidence": np.nanmean(incidence_boot[i]),
        "Incidence CI Low": inc_ci_low,
        "Incidence CI High": inc_ci_high,

        "TP": tp_list[i],
        "FP": fp_list[i],
        "TN": tn_list[i],
        "FN": fn_list[i],

        "At Risk": at_risk_list[i],
    })

c:\Users\jenan\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1409: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\1243856042.py:32: RuntimeWarning: Mean of empty slice
  "PPV": np.nanmean(ppv_boot[i]),
C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\1243856042.py:20: RuntimeWarning: Mean of empty slice
  "AUC": np.nanmean(auc_boot[i]),
C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\1243856042.py:24: RuntimeWarning: Mean of empty slice
  "Sensitivity": np.nanmean(sens_boot[i]),


In [122]:
# # get df of results
# results_df = pd.DataFrame({
#     'Time Point': time_point,
#     'Incidence': incidence,
#     'TP': tp,
#     'FP': fp,
#     'TN': tn,
#     'FN': fn,
#     'AUC': auc,
#     'Sensitivity': sensitivity,
#     'Specificity': specificity,
#     'PPV': ppv,
#     'NPV': npv,
#     'At Risk': at_risk
# })

In [123]:
results_df = pd.DataFrame(summary)

In [124]:
# reorder results df
results_df.sort_values(by='Time Point', ascending=True, inplace=False)
results_df

,Time Point,AUC,AUC CI Low,AUC CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,Specificity CI Low,Specificity CI High,...,NPV CI Low,NPV CI High,Incidence,Incidence CI Low,Incidence CI High,TP,FP,TN,FN,At Risk
0,4,0.698233,0.679275,0.721018,0.477313,0.438720,0.521974,0.919154,0.916149,0.922328,...,0.988568,0.990922,1.788300,1.642759,1.929895,267,2475,28162,292,31196
1,5,0.577006,0.560141,0.593706,0.171883,0.138336,0.205159,0.982129,0.980577,0.983580,...,0.984008,0.986731,1.721541,1.592084,1.859074,89,530,29129,431,30179
2,6,0.558104,0.544340,0.572240,0.126987,0.099576,0.155763,0.989222,0.988126,0.990411,...,0.984083,0.986716,1.650811,1.505391,1.808536,61,307,28243,418,29029
3,7,0.541591,0.529477,0.554672,0.089380,0.065006,0.116256,0.993802,0.992887,0.994664,...,0.983820,0.986707,1.602880,1.443381,1.753114,40,168,27169,405,27782
4,8,0.527535,0.516490,0.539631,0.060796,0.038554,0.084722,0.994275,0.993301,0.995167,...,0.983316,0.986248,1.599879,1.456714,1.758623,26,150,25925,397,26498
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10137,10141,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10138,10142,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10139,10143,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10140,10144,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1


In [125]:
# save
#results_df.to_csv(os.path.join(data_save_path, 'Time_dependent_metric_results.csv'), index = False)

In [126]:
# subset df for 3.5 days, set legend to be in days
subset_results_df = results_df[results_df['Time Point'] <= 336]
subset_results_df['Time Point Days'] = subset_results_df['Time Point'] / 96

C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\2903172409.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_results_df['Time Point Days'] = subset_results_df['Time Point'] / 96


In [127]:
days = [8, 48, 96, 144, 192, 240, 288, 336]
table_df = subset_results_df[subset_results_df['Time Point'].isin(days)][['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']]
table_df[['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']] = table_df[['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']].round(2)
table_df = table_df.T

In [128]:
subset_results_df

,Time Point,AUC,AUC CI Low,AUC CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,Specificity CI Low,Specificity CI High,...,NPV CI High,Incidence,Incidence CI Low,Incidence CI High,TP,FP,TN,FN,At Risk,Time Point Days
0,4,0.698233,0.679275,0.721018,0.477313,0.438720,0.521974,0.919154,0.916149,0.922328,...,0.990922,1.788300,1.642759,1.929895,267,2475,28162,292,31196,0.041667
1,5,0.577006,0.560141,0.593706,0.171883,0.138336,0.205159,0.982129,0.980577,0.983580,...,0.986731,1.721541,1.592084,1.859074,89,530,29129,431,30179,0.052083
2,6,0.558104,0.544340,0.572240,0.126987,0.099576,0.155763,0.989222,0.988126,0.990411,...,0.986716,1.650811,1.505391,1.808536,61,307,28243,418,29029,0.062500
3,7,0.541591,0.529477,0.554672,0.089380,0.065006,0.116256,0.993802,0.992887,0.994664,...,0.986707,1.602880,1.443381,1.753114,40,168,27169,405,27782,0.072917
4,8,0.527535,0.516490,0.539631,0.060796,0.038554,0.084722,0.994275,0.993301,0.995167,...,0.986248,1.599879,1.456714,1.758623,26,150,25925,397,26498,0.083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,332,0.498417,0.497506,0.499150,0.000000,0.000000,0.000000,0.996833,0.995012,0.998300,...,0.998162,0.367338,0.183198,0.575766,0,12,3795,14,3821,3.458333
329,333,0.498552,0.497628,0.499342,0.000000,0.000000,0.000000,0.997104,0.995256,0.998683,...,0.997895,0.390706,0.210029,0.577579,0,11,3783,15,3809,3.468750
330,334,0.498642,0.497884,0.499471,0.000000,0.000000,0.000000,0.997283,0.995768,0.998942,...,0.998149,0.367869,0.184356,0.593231,0,10,3773,14,3797,3.479167
331,335,0.498830,0.498011,0.499602,0.000000,0.000000,0.000000,0.997661,0.996023,0.999204,...,0.997885,0.397098,0.211026,0.606700,0,9,3767,15,3791,3.489583


In [129]:
subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')

C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\724104491.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')
C:\Users\jenan\AppData\Local\Temp\ipykernel_22496\724104491.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')


In [130]:
def check_nans_in_columns(df, columns):
    for col in columns:
        n_missing = df[col].isna().sum()
        print(f"Column '{col}' has {n_missing} missing values (NaNs)")
cols_to_check = ['Time Point Days', 'Incidence', 'AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV']
check_nans_in_columns(subset_results_df, cols_to_check)

Column 'Time Point Days' has 0 missing values (NaNs)
Column 'Incidence' has 0 missing values (NaNs)
Column 'AUC' has 0 missing values (NaNs)
Column 'Sensitivity' has 0 missing values (NaNs)
Column 'Specificity' has 0 missing values (NaNs)
Column 'PPV' has 0 missing values (NaNs)
Column 'NPV' has 0 missing values (NaNs)


In [131]:
subset_results_df_to_smooth = subset_results_df.copy()

In [132]:
# LOWESS SMOOTHING FOR EACH COL BASED ON TIME POINT DAYS X
def apply_lowess_multiple(df, x_col, y_cols, frac):
    for col in y_cols:
        smoothed = lowess(df[col], df[x_col], frac)
        df[col] = smoothed[:, 1]
    return df
subset_results_df_smoothed = apply_lowess_multiple(subset_results_df_to_smooth, 'Time Point Days', ['Incidence', 'AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV'], 0.15)

In [ ]:
# plot without CIs
# import matplotlib.pyplot as plt
# from matplotlib.gridspec import GridSpec
# import os

# # Base font sizes
# label_fontsize = 16
# title_fontsize = 18
# tick_fontsize = 14
# table_fontsize = 16

# # Figure size: wider to match table
# fig = plt.figure(figsize=(16, 20))

# # GridSpec: 4 rows, 2 columns
# # height_ratios: 3 rows of plots, 1 row for table (taller)
# # width_ratios: both columns equal width
# gs = GridSpec(4, 2, figure=fig, height_ratios=[1, 1, 1, 1.5], width_ratios=[1, 1], hspace=0.4, wspace=0.3)

# # ---- Create axes ----
# # Left column plots
# ax_inc = fig.add_subplot(gs[0, 0])
# ax_sens = fig.add_subplot(gs[1, 0], sharex=ax_inc)
# ax_spec = fig.add_subplot(gs[2, 0], sharex=ax_inc)

# # Right column plots
# ax_ppv = fig.add_subplot(gs[0, 1], sharex=ax_inc)
# ax_npv = fig.add_subplot(gs[1, 1], sharex=ax_inc)
# ax_auc = fig.add_subplot(gs[2, 1], sharex=ax_inc)

# # Bottom table spanning both columns
# ax_table = fig.add_subplot(gs[3, :])

# # ---- Left column ----
# ax_inc.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['Incidence'])
# ax_inc.set_ylabel('Incidence', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Incidence'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_inc.set_title(f'Incidence (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_inc.tick_params(axis='both', labelsize=tick_fontsize)

# ax_sens.plot(subset_results_df_smoothed['Time Point Days'],
#              subset_results_df_smoothed['Sensitivity'])
# ax_sens.set_ylabel('Sensitivity', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Sensitivity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_sens.set_title(f'Sensitivity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_sens.tick_params(axis='both', labelsize=tick_fontsize)

# ax_spec.plot(subset_results_df_smoothed['Time Point Days'],
#              subset_results_df_smoothed['Specificity'])
# ax_spec.set_ylabel('Specificity', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Specificity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_spec.set_title(f'Specificity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_spec.tick_params(axis='both', labelsize=tick_fontsize)

# # ---- Right column ----
# ax_ppv.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['PPV'])
# ax_ppv.set_ylabel('PPV', fontsize=label_fontsize)
# ax_ppv.set_ylim(0.0, 0.3)
# weighted_avg = (subset_results_df_smoothed['PPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_ppv.set_title(f'PPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_ppv.tick_params(axis='both', labelsize=tick_fontsize)

# ax_npv.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['NPV'])
# ax_npv.set_ylabel('NPV', fontsize=label_fontsize)
# ax_npv.set_ylim(0.5, 1.0)
# weighted_avg = (subset_results_df_smoothed['NPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_npv.set_title(f'NPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_npv.tick_params(axis='both', labelsize=tick_fontsize)

# ax_auc.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['AUC'])
# ax_auc.set_ylabel('AUC', fontsize=label_fontsize)
# ax_auc.set_ylim(0.5, 1.0)
# q1 = subset_results_df_smoothed['AUC'].quantile(0.25)
# q3 = subset_results_df_smoothed['AUC'].quantile(0.75)
# weighted_avg = (subset_results_df_smoothed['AUC'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_auc.set_title(f'AUC (weighted avg: {round(weighted_avg, 2)}, IQR: [{round(q1, 2)}, {round(q3, 2)}])', fontsize=title_fontsize)
# ax_auc.tick_params(axis='both', labelsize=tick_fontsize)

# # Shared x-label for bottom plots
# ax_spec.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
# ax_auc.set_xlabel('Time Point (Days)', fontsize=label_fontsize)

# # ---- Bottom table ----
# ax_table.axis("off")
# table = ax_table.table(
#     cellText=table_df.values,
#     rowLabels=['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN'],
#     loc="center",
#     cellLoc='center',
#     rowLoc='center',
#     colWidths=[0.15] * table_df.shape[1]
# )

# # Table font and row height
# table.auto_set_font_size(False)
# table.set_fontsize(table_fontsize)
# for key, cell in table.get_celld().items():
#     cell.set_height(0.09)  # increase row height
#     cell.set_linewidth(1.5)  # thicker grid lines

# # Adjust layout
# plt.tight_layout()
# plt.savefig(os.path.join(data_save_path, 'metric_figure_smoothed.png'), bbox_inches="tight")
# plt.show()

In [ ]:
# plot with CIs
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

label_fontsize = 16
title_fontsize = 18
tick_fontsize = 14
table_fontsize = 16

fig = plt.figure(figsize=(16, 20))

gs = GridSpec(
    4, 2,
    figure=fig,
    height_ratios=[1, 1, 1, 1.5],
    width_ratios=[1, 1],
    hspace=0.4,
    wspace=0.3
)

ax_inc = fig.add_subplot(gs[0, 0])
ax_sens = fig.add_subplot(gs[1, 0], sharex=ax_inc)
ax_spec = fig.add_subplot(gs[2, 0], sharex=ax_inc)

ax_ppv = fig.add_subplot(gs[0, 1], sharex=ax_inc)
ax_npv = fig.add_subplot(gs[1, 1], sharex=ax_inc)
ax_auc = fig.add_subplot(gs[2, 1], sharex=ax_inc)

ax_table = fig.add_subplot(gs[3, :])

x = subset_results_df_smoothed['Time Point Days']

ax_inc.plot(x, subset_results_df_smoothed['Incidence'], color='C0')
ax_inc.fill_between(
    x,
    subset_results_df_smoothed['Incidence CI Low'],
    subset_results_df_smoothed['Incidence CI High'],
    alpha=0.2,
    color='C0'
)

weighted_avg = (subset_results_df_smoothed['Incidence'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_inc.set_ylabel('Incidence', fontsize=label_fontsize)
ax_inc.set_title(f'Incidence (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
ax_inc.tick_params(axis='both', labelsize=tick_fontsize)

ax_sens.plot(x, subset_results_df_smoothed['Sensitivity'], color='C1')
ax_sens.fill_between(
    x,
    subset_results_df_smoothed['Sensitivity CI Low'],
    subset_results_df_smoothed['Sensitivity CI High'],
    alpha=0.2,
    color='C1'
)

weighted_avg = (subset_results_df_smoothed['Sensitivity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_sens.set_ylabel('Sensitivity', fontsize=label_fontsize)
ax_sens.set_title(f'Sensitivity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
ax_sens.tick_params(axis='both', labelsize=tick_fontsize)

ax_spec.plot(x, subset_results_df_smoothed['Specificity'], color='C2')
ax_spec.fill_between(
    x,
    subset_results_df_smoothed['Specificity CI Low'],
    subset_results_df_smoothed['Specificity CI High'],
    alpha=0.2,
    color='C2'
)

weighted_avg = (subset_results_df_smoothed['Specificity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_spec.set_ylabel('Specificity', fontsize=label_fontsize)
ax_spec.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
ax_spec.set_title(f'Specificity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
ax_spec.tick_params(axis='both', labelsize=tick_fontsize)

ax_ppv.plot(x, subset_results_df_smoothed['PPV'], color='C3')
ax_ppv.fill_between(
    x,
    subset_results_df_smoothed['PPV CI Low'],
    subset_results_df_smoothed['PPV CI High'],
    alpha=0.2,
    color='C3'
)

weighted_avg = (subset_results_df_smoothed['PPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_ppv.set_ylabel('PPV', fontsize=label_fontsize)
ax_ppv.set_ylim(0.0, 0.3)
ax_ppv.set_title(f'PPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
ax_ppv.tick_params(axis='both', labelsize=tick_fontsize)

ax_npv.plot(x, subset_results_df_smoothed['NPV'], color='C4')
ax_npv.fill_between(
    x,
    subset_results_df_smoothed['NPV CI Low'],
    subset_results_df_smoothed['NPV CI High'],
    alpha=0.2,
    color='C4'
)

weighted_avg = (subset_results_df_smoothed['NPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_npv.set_ylabel('NPV', fontsize=label_fontsize)
ax_npv.set_ylim(0.5, 1.0)
ax_npv.set_title(f'NPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
ax_npv.tick_params(axis='both', labelsize=tick_fontsize)

ax_auc.plot(x, subset_results_df_smoothed['AUC'], color='C5')
ax_auc.fill_between(
    x,
    subset_results_df_smoothed['AUC CI Low'],
    subset_results_df_smoothed['AUC CI High'],
    alpha=0.2,
    color='C5'
)

q1 = subset_results_df_smoothed['AUC'].quantile(0.25)
q3 = subset_results_df_smoothed['AUC'].quantile(0.75)

weighted_avg = (subset_results_df_smoothed['AUC'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

ax_auc.set_ylabel('AUC', fontsize=label_fontsize)
ax_auc.set_ylim(0.5, 1.0)
ax_auc.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
ax_auc.set_title(
    f'AUC (weighted avg: {round(weighted_avg, 2)}, IQR: [{round(q1, 2)}, {round(q3, 2)}])',
    fontsize=title_fontsize
)
ax_auc.tick_params(axis='both', labelsize=tick_fontsize)

ax_table.axis("off")

table = ax_table.table(
    cellText=table_df.values,
    rowLabels=['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN'],
    loc="center",
    cellLoc='center',
    rowLoc='center',
    colWidths=[0.15] * table_df.shape[1]
)

table.auto_set_font_size(False)
table.set_fontsize(table_fontsize)

for key, cell in table.get_celld().items():
    cell.set_height(0.09)
    cell.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(
    os.path.join(data_save_path, 'metric_figure_smoothed.png'),
    bbox_inches="tight"
)
plt.show()

In [ ]:
table_df.to_csv(os.path.join(data_save_path, 'table.csv'))

In [ ]:
# just metrics
df = subset_results_df_smoothed

def weighted_avg(x, w):
    return np.sum(x * w) / np.sum(w)

def iqr(x):
    return np.nanpercentile(x, 75) - np.nanpercentile(x, 25)

metrics = {
    "Incidence": df["Incidence"],
    "Sensitivity": df["Sensitivity"],
    "Specificity": df["Specificity"],
    "PPV": df["PPV"],
    "NPV": df["NPV"],
    "AUC": df["AUC"],
}

weights = df["At Risk"]

summary_rows = []

for name, series in metrics.items():
    summary_rows.append({
        "Metric": name,
        "Weighted Mean": weighted_avg(series.values, weights.values),
        "Mean": np.nanmean(series),
        "Median": np.nanmedian(series),
        "IQR": iqr(series),
        "Q1": np.nanpercentile(series, 25),
        "Q3": np.nanpercentile(series, 75),
        "Min": np.nanmin(series),
        "Max": np.nanmax(series),
    })

summary_df = pd.DataFrame(summary_rows)

output_path = os.path.join(data_save_path, "metric_summary.csv")
summary_df.to_csv(output_path, index=False)